# 20. LLM 后训练中的 RL 与 PO 体系
大语言模型后训练的核心问题不是“继续做一次预训练”，而是把模型从“会补全文本”推进到“更符合人类偏好、任务目标和系统约束”。这一步通常被称为 post-training，包括监督微调（SFT）、奖励建模、在线强化学习和离线偏好优化（Preference Optimization, PO）。

截至 **2026-03-17**，公开文献与主流开源实践已经形成两条相互交织的主线：

- **RL 主线**：PPO、RLOO、GRPO、GSPO、SAPO、GiGPO 等，强调从在线采样中构造策略改进；
- **PO 主线**：DPO、ORPO、KTO、SimPO、chiPO 等，强调直接从偏好数据构造可优化目标，不显式训练奖励模型或 critic。

本 notebook 的目标是把这些方法放进统一视角：它们都在回答同一个问题，即“如何稳定地把偏好信号转化为策略更新”。

## LLM 后训练中的策略优化问题的形式化定义

            对大语言模型而言，策略就是条件分布 $\pi_\theta(y\mid x)$，其中 $x$ 是提示词，$y$ 是完整响应。后训练的目标不再只是最大化下一个 token 的似然，而是最大化某种**序列级偏好目标**。如果有奖励函数 $r(x,y)$，可以写成
$$
\max_\pi \ \mathbb{E}_{x\sim\mathcal{D}, y\sim\pi(\cdot\mid x)}[r(x,y)] - \beta \, D(\pi \,\|\, \pi_{\mathrm{ref}})
$$
其中 $\pi_{\mathrm{ref}}$ 通常是 SFT 模型，$D$ 可以是 KL 或其他散度，$\beta$ 控制“向奖励靠拢”和“不要偏离参考模型太远”之间的平衡。

## 输入、输出与参数化方式

            输入是提示 $x$、候选响应 $y$、偏好标签或奖励分数；输出仍然是语言模型参数 $\theta$。不同方法的差别在于：

- 学习信号是**成对偏好**、**分组相对分数**还是**在线采样回报**；
- 更新单位是**token 级**、**序列级**还是**组级**；
- 是否显式训练 reward model / critic；
- 是否需要 old policy / reference policy / trust region。

## 结构分解与信息流

            后训练链路通常分为四段：

1. 预训练模型提供通用语言能力；
2. SFT 把输出分布拉到任务相关区域；
3. 偏好数据、规则反馈或可验证奖励提供“好坏排序”；
4. RL 或 PO 把这些排序/奖励转换为参数更新。

PPO、GRPO、GSPO、SAPO、GiGPO 更像“在线策略优化器”；DPO、ORPO、KTO、SimPO、chiPO 更像“离线偏好目标构造器”；GDPO 则试图把偏好优化与分布级多样性建模连接起来。

## 优化目标与训练机制

            本体系的关键目标函数包括：

- **PPO / RLHF**：
  $$
  \mathcal{L}_{\text{PPO}} = \mathbb{E}\left[\min\left(r_t \hat{A}_t, \operatorname{clip}(r_t, 1-\epsilon, 1+\epsilon)\hat{A}_t\right)\right] - \beta \,\mathrm{KL}(\pi_\theta \,\|\, \pi_{\mathrm{ref}})
  $$

- **DPO**：
  $$
  \mathcal{L}_{\text{DPO}} =
  -\log \sigma\left(
  \beta \left[
  \log \frac{\pi_\theta(y_w\mid x)}{\pi_\theta(y_l\mid x)}
  -
  \log \frac{\pi_{\mathrm{ref}}(y_w\mid x)}{\pi_{\mathrm{ref}}(y_l\mid x)}
  \right]\right)
  $$

- **GRPO**：对同一提示采样一组响应，利用组内相对奖励构造归一化优势
  $$
  A_i = \frac{r_i - \mathrm{mean}(r)}{\mathrm{std}(r)+\varepsilon}
  $$
  再做 PPO 风格更新，但不显式训练 value model。

- **GSPO / SAPO / GiGPO**：继续改造组式策略优化中的 ratio 控制与 credit assignment。

## 计算复杂度、统计性质与工程代价

            LLM 后训练的难点不只是优化器本身，而是信号噪声、长度偏差、奖励可被投机利用、KL 漂移、样本成本和工程吞吐共同作用的结果。一个看似更“先进”的目标函数，如果不能控制长度偏置、难以并行采样或对 batch 组成极端敏感，实际效果可能反而更差。

因此评价现代 PO / RL 方法时，必须同时看：稳定性、样本效率、是否依赖 critic、是否需要在线采样、是否保留多样性、是否容易 reward hacking。

## 与相邻模型的差异

PPO 是 RLHF 的经典基线，但需要 value model、old policy 和较重的在线训练链路。DPO 家族绕过 critic，适合离线偏好数据；GRPO 通过组内相对比较去掉 value model，在大模型在线训练中更高效；GSPO 和 SAPO 聚焦 ratio 控制的稳定性；GiGPO 进一步把信用分配扩展到 group-in-group 结构；GDPO 则强调分布级多样性与多模态高质量输出，而不是单峰模式塌缩。

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(14, 5))
ax.axis("off")
ax.set_xlim(0, 14)
ax.set_ylim(0, 5)

stages = [
    (1.2, 2.5, 1.8, 1.1, "#4C78A8", "pretraining"),
    (4.0, 2.5, 1.8, 1.1, "#F58518", "SFT"),
    (6.8, 2.5, 2.0, 1.1, "#E45756", "preference / reward"),
    (9.8, 2.5, 2.1, 1.1, "#72B7B2", "RL or PO"),
    (12.5, 2.5, 1.8, 1.1, "#54A24B", "aligned model"),
]
for x, y, w, h, color, text in stages:
    rect = plt.Rectangle((x - w / 2, y - h / 2), w, h, facecolor=color, edgecolor="black")
    ax.add_patch(rect)
    ax.text(x, y, text, ha="center", va="center", fontsize=11)
for i in range(len(stages) - 1):
    ax.annotate("", xy=(stages[i + 1][0] - stages[i + 1][2] / 2, 2.5), xytext=(stages[i][0] + stages[i][2] / 2, 2.5),
                arrowprops=dict(arrowstyle="->", lw=1.8))
ax.text(6.8, 4.0, "offline PO uses preference pairs; online RL uses sampled rollouts plus reward / rule signals", ha="center", fontsize=11)
ax.set_title("LLM Post-Training Pipeline")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(13, 5.5))
ax.axis("off")
ax.set_xlim(0, 13)
ax.set_ylim(0, 5.5)

ax.add_patch(plt.Rectangle((0.9, 2.2), 1.8, 1.2, facecolor="#4C78A8", edgecolor="black"))
ax.text(1.8, 2.8, "prompt x", ha="center", va="center", fontsize=11)

group_y = [4.4, 3.2, 2.0, 0.8]
for idx, y in enumerate(group_y, start=1):
    ax.add_patch(plt.Rectangle((3.5, y), 2.2, 0.7, facecolor="#F58518", edgecolor="black"))
    ax.text(4.6, y + 0.35, f"sampled response y{idx}", ha="center", va="center", fontsize=10)

ax.add_patch(plt.Rectangle((7.4, 2.2), 2.3, 1.2, facecolor="#E45756", edgecolor="black"))
ax.text(8.55, 2.8, "group reward\nnormalize -> A_i", ha="center", va="center", fontsize=11)

ax.add_patch(plt.Rectangle((11.1, 2.2), 2.0, 1.2, facecolor="#54A24B", edgecolor="black"))
ax.text(12.1, 2.8, "policy update", ha="center", va="center", fontsize=11)

ax.annotate("", xy=(3.5, 2.8), xytext=(2.7, 2.8), arrowprops=dict(arrowstyle="->", lw=1.8))
for y in group_y:
    ax.annotate("", xy=(7.4, 2.8), xytext=(5.7, y + 0.35), arrowprops=dict(arrowstyle="->", lw=1.4))
ax.annotate("", xy=(11.1, 2.8), xytext=(9.7, 2.8), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.text(6.6, 4.9, "GRPO / GSPO / SAPO / GiGPO start from grouped rollouts but differ in credit assignment and ratio control", ha="center", fontsize=10.5)
ax.set_title("Grouped Policy Optimization for LLM Post-Training")
plt.show()

## 统一视角：从奖励正则化到偏好优化

许多方法表面形式不同，但可以放进同一个奖励正则化框架：
$$
\pi^\star(y\mid x) \propto \pi_{\mathrm{ref}}(y\mid x)\exp\left(\frac{1}{\beta}r(x,y)\right).
$$
这表示最优策略会在参考模型分布上对高奖励响应做指数重加权。差别在于：

- PPO 用在线采样 + advantage 近似这个过程；
- DPO 把偏好对 $(y_w, y_l)$ 直接转成 logit 差的分类目标；
- ORPO 把偏好项叠加到有监督目标上；
- KTO 用“值得保留 / 值得压低”的 Kahneman-Tversky 风格效用构造目标；
- SimPO 去掉参考模型，直接用当前策略的平均 log-prob margin 做比较；
- chiPO 则用更稳健的散度形式缓和 KL 型目标的极端梯度。

## 重点算法详解

### 1. PPO 与 RLOO

PPO 是 RLHF 的标准起点。它依赖 reward model 与 value model，通过 old policy ratio 限制策略偏移。RLOO（REINFORCE Leave-One-Out）则在不训练 critic 的前提下，用同组样本的 leave-one-out baseline 估计优势，降低训练链路复杂度，常被视为从 PPO 走向 group-based RL 的过渡方法。

### 2. GRPO

GRPO（Group Relative Policy Optimization，2024-02-06）对同一提示采样一组候选响应，通过组内相对奖励构造标准化优势，从而移除 value model。它特别适合“同提示下相对质量可比较”的任务，例如数学推理、代码生成、可验证答案任务。GRPO 的关键优点是工程简化，但它对组内奖励方差、长度偏差和 token 级 ratio 的累积噪声较敏感。

### 3. GDPO

GDPO（GFlowNet-based Distributional Preference Optimization，EMNLP 2024）不把目标理解成“找唯一最优输出”，而是试图学习**与奖励成比例的高质量分布**。这使它在多峰偏好任务中更强调多样性保持，特别适合存在多种等价高质量答案的场景。可以把它理解成“把偏好优化从点估计推进到分布估计”。

### 4. GSPO

GSPO（Generalized / Group Sequence Policy Optimization，2025-07-24 的公开论文）把更新重心从 token 级 ratio 推向**序列级 ratio**。这背后的判断是：LLM 偏好通常是序列级质量，不应被单个 token 的极端比例扰动主导。序列级 ratio 更接近真正的响应级偏好信号，因此在长序列场景下更稳定。公开资料显示，GSPO 已成为 2025 年后 group-based LLM RL 中的重要方向之一。

### 5. SAPO

SAPO（Softmax / Smoothed Approximate Policy Optimization，2025-11-25 arXiv）针对 PPO / GRPO 中硬 clipping 带来的不可导拐点与梯度截断问题，引入更平滑的 ratio 抑制机制。它的核心思想不是彻底放弃 trust region，而是把“超过阈值就硬截断”改成“偏离越大，惩罚越平滑地增强”。这在高方差序列奖励下更容易获得连续稳定的梯度。

### 6. GiGPO

GiGPO（Group-in-Group Policy Optimization，2025-05-15）把 group-based RL 从“每个 prompt 一个组”进一步推进到“组中再分组”的层级信用分配。对于多轮对话、agent trajectory、带工具调用的复杂流程任务，它允许同时建模全局轨迹质量与局部关键步骤质量，从而缓解“整条轨迹一个总分”过于粗糙的问题。

## 适用场景对比

| 方法 | 数据类型 | 是否在线采样 | 是否需要 critic / reward model | 更新粒度 | 典型优势 | 主要风险 |
| --- | --- | --- | --- | --- | --- | --- |
| PPO | 在线轨迹 + reward | 是 | 通常需要 reward model 与 value model | token / trajectory | 通用 RLHF 基线成熟 | 工程链路重，训练成本高 |
| RLOO | 在线组采样 | 是 | 不需要 critic | trajectory / group | 简化 PPO | 方差仍可能较大 |
| DPO | 离线偏好对 | 否 | 不需要 reward model | pairwise sequence | 实现简单、训练稳定 | 受参考模型与数据偏差影响 |
| ORPO | 离线偏好对 + SFT | 否 | 不需要 reward model | pairwise sequence | 一步联合训练 | 目标耦合较强 |
| KTO | 离线正负示例 | 否 | 不需要 reward model | sequence | 不要求严格成对数据 | 效用刻画依赖设计 |
| SimPO | 离线偏好对 | 否 | 不需要 reference model | sequence | 更轻量 | 对 margin 与长度敏感 |
| GRPO | 在线组采样 | 是 | 不需要 value model | group relative | 工程更轻、适合可验证任务 | 组方差与 token ratio 噪声 |
| GDPO | 离线 / 近离线偏好分布 | 可选 | 不强调传统 critic | distribution | 保持多样性、避免单峰塌缩 | 实现与解释门槛较高 |
| GSPO | 在线组采样 | 是 | 不需要 value model | sequence-level ratio | 长序列更稳定 | 仍需高质量组内奖励 |
| SAPO | 在线组采样 | 是 | 不需要 value model | smooth ratio control | 梯度更平滑 | 新方法，工程经验仍在积累 |
| GiGPO | 在线层级轨迹 | 是 | 不需要传统 critic | group-in-group | 更细粒度信用分配 | 管线更复杂、标注更难 |

In [ ]:
# 兼容当前 Windows 环境：把临时目录固定到用户目录下的 ASCII 路径，
# 避免 scipy / sklearn 在中文工作目录下寻找临时文件时报错。
from pathlib import Path
import os
import warnings

temp_root = Path(os.environ.get("ML_DL_TMP", str(Path.home() / ".ml_dl_notebook_tmp")))
temp_root.mkdir(exist_ok=True)
os.environ["TMP"] = str(temp_root)
os.environ["TEMP"] = str(temp_root)

warnings.filterwarnings("ignore")

import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
random.seed(42)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]



print("临时目录:", temp_root)

In [ ]:
def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / exp_x.sum(axis=axis, keepdims=True)

prompts = [f"prompt_{i}" for i in range(1, 6)]
responses = [f"resp_{j}" for j in range(1, 6)]

reward_matrix = np.array(
    [
        [0.95, 0.82, 0.40, 0.35, 0.20],
        [0.60, 0.91, 0.88, 0.30, 0.10],
        [0.15, 0.25, 0.92, 0.89, 0.55],
        [0.78, 0.76, 0.20, 0.18, 0.12],
        [0.25, 0.42, 0.73, 0.85, 0.81],
    ]
)
ref_logits = np.array(
    [
        [1.4, 1.0, 0.2, -0.2, -0.5],
        [0.6, 1.1, 0.9, -0.1, -0.4],
        [-0.3, 0.1, 1.0, 0.9, 0.5],
        [1.0, 0.9, 0.0, -0.2, -0.4],
        [-0.1, 0.2, 0.7, 1.1, 0.8],
    ]
)
policy_logits = ref_logits + np.array(
    [
        [0.10, -0.05, 0.20, -0.10, -0.15],
        [-0.10, 0.05, 0.08, -0.05, -0.08],
        [-0.05, -0.03, 0.12, 0.05, 0.02],
        [0.05, 0.04, -0.10, -0.08, -0.10],
        [-0.08, -0.02, 0.04, 0.10, 0.06],
    ]
)

pi_ref = softmax(ref_logits, axis=1)
pi_cur = softmax(policy_logits, axis=1)

reward_df = pd.DataFrame(reward_matrix, index=prompts, columns=responses)
reward_df

In [ ]:
plt.figure(figsize=(9, 5))
sns.heatmap(reward_df, annot=True, cmap="YlGnBu", fmt=".2f")
plt.title("Toy Prompt-Response Reward Matrix")
plt.show()

In [ ]:
beta = 3.0
winner_idx = reward_matrix.argmax(axis=1)
loser_idx = reward_matrix.argmin(axis=1)

dpo_records = []
for i, prompt in enumerate(prompts):
    y_w = winner_idx[i]
    y_l = loser_idx[i]
    delta_model = np.log(pi_cur[i, y_w] / pi_cur[i, y_l])
    delta_ref = np.log(pi_ref[i, y_w] / pi_ref[i, y_l])
    margin = beta * (delta_model - delta_ref)
    loss = -np.log(1 / (1 + np.exp(-margin)))
    dpo_records.append(
        {
            "prompt": prompt,
            "winner": responses[y_w],
            "loser": responses[y_l],
            "dpo_margin": margin,
            "dpo_loss": loss,
        }
    )
dpo_df = pd.DataFrame(dpo_records)
dpo_df

In [ ]:
group_advantages = []
gdpo_targets = []
for i, prompt in enumerate(prompts):
    rewards = reward_matrix[i]
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-6)
    for response, reward, adv in zip(responses, rewards, advantages):
        group_advantages.append(
            {"prompt": prompt, "response": response, "reward": reward, "grpo_advantage": adv}
        )

    reward_regularized = pi_ref[i] * np.exp(beta * rewards)
    reward_regularized = reward_regularized / reward_regularized.sum()
    for response, prob in zip(responses, reward_regularized):
        gdpo_targets.append({"prompt": prompt, "response": response, "gdpo_target_prob": prob})

grpo_df = pd.DataFrame(group_advantages)
gdpo_df = pd.DataFrame(gdpo_targets)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
sns.barplot(
    data=grpo_df[grpo_df["prompt"] == "prompt_1"],
    x="response",
    y="grpo_advantage",
    palette="coolwarm",
    ax=axes[0],
)
axes[0].set_title("GRPO Normalized Group Advantages")

sns.barplot(
    data=gdpo_df[gdpo_df["prompt"] == "prompt_2"],
    x="response",
    y="gdpo_target_prob",
    color="#54A24B",
    ax=axes[1],
)
axes[1].set_title("GDPO-style Reward-Proportional Target Distribution")
plt.tight_layout()
plt.show()

In [ ]:
ratios = np.linspace(0.4, 1.8, 240)
eps = 0.2
grpo_update = np.minimum(ratios * 1.0, np.clip(ratios, 1 - eps, 1 + eps) * 1.0)
gspo_update = np.minimum(ratios * 1.0, np.clip(ratios, 1 - eps, 1 + eps) * 1.0)
sapo_gate = 1 / np.cosh((ratios - 1.0) / 0.18) ** 2
sapo_update = ratios * sapo_gate

ratio_df = pd.DataFrame(
    {
        "ratio": ratios,
        "GRPO / PPO clipped": grpo_update,
        "GSPO sequence-level clipped": gspo_update,
        "SAPO smooth gate": sapo_update,
    }
)
ratio_long = ratio_df.melt(id_vars="ratio", var_name="method", value_name="update_strength")

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(data=ratio_long, x="ratio", y="update_strength", hue="method")
plt.axvline(1 - eps, color="gray", linestyle="--")
plt.axvline(1 + eps, color="gray", linestyle="--")
plt.title("Trust-Region Control: Hard Clip vs Smooth Gate")
plt.ylabel("effective update strength")
plt.show()

In [ ]:
token_ratios = np.array([1.02, 1.01, 0.99, 1.03, 1.65, 0.98, 1.00, 1.01])
seq_ratio = token_ratios.prod() ** (1 / len(token_ratios))
ratio_compare_df = pd.DataFrame(
    {
        "unit": [f"token_{i}" for i in range(1, len(token_ratios) + 1)] + ["sequence_geomean"],
        "ratio": list(token_ratios) + [seq_ratio],
    }
)
ratio_compare_df

In [ ]:
plt.figure(figsize=(10, 4.5))
sns.barplot(data=ratio_compare_df, x="unit", y="ratio", palette="mako")
plt.axhline(1.0, color="gray", linestyle="--")
plt.title("Why GSPO Prefers Sequence-Level Ratio Control")
plt.xticks(rotation=45)
plt.show()

In [ ]:
trajectories = pd.DataFrame(
    {
        "trajectory": ["traj_A"] * 4 + ["traj_B"] * 4 + ["traj_C"] * 4,
        "stage": ["plan", "tool", "verify", "final"] * 3,
        "local_score": [0.9, 0.4, 0.8, 0.85, 0.5, 0.95, 0.45, 0.60, 0.8, 0.82, 0.78, 0.88],
        "global_score": [0.84] * 4 + [0.63] * 4 + [0.82] * 4,
    }
)
trajectories["global_adv"] = trajectories["global_score"] - trajectories["global_score"].mean()
trajectories["local_adv"] = trajectories.groupby("stage")["local_score"].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-6)
)
trajectories["gigpo_credit"] = 0.6 * trajectories["global_adv"] + 0.4 * trajectories["local_adv"]
trajectories

In [ ]:
gigpo_pivot = trajectories.pivot(index="trajectory", columns="stage", values="gigpo_credit")
plt.figure(figsize=(8, 4.8))
sns.heatmap(gigpo_pivot, annot=True, cmap="RdBu_r", center=0.0, fmt=".2f")
plt.title("GiGPO-style Hierarchical Credit Assignment")
plt.show()

## 实践判断标准

在真实 LLM 后训练中，不应只问“某算法在榜单上是否更高”，而应同时问：

- 它依赖的是离线偏好数据，还是昂贵的在线采样？
- 它是否需要 reward model / critic，工程链路是否能承担？
- 它的更新单位是 token、sequence 还是 group，是否与任务奖励粒度一致？
- 它是否显著放大长度偏差、参考模型偏差或奖励作弊风险？
- 它是适合单峰最优答案任务，还是更强调多样性与多模态高质量分布？

这也是为什么当前主流实践往往不是“只用一个算法”，而是根据任务类型在 SFT、DPO 类方法和在线 group-based RL 方法之间做组合。

## 主要资料

- InstructGPT / RLHF（NeurIPS 2022）:
  https://papers.nips.cc/paper_files/paper/2022/hash/b1efde53be364a73914f58805a001731-Abstract-Conference.html
- PPO（2017）: https://arxiv.org/abs/1707.06347
- DPO（2023-05-29）: https://arxiv.org/abs/2305.18290
- KTO（2024-02-02）: https://arxiv.org/abs/2402.01306
- GRPO（2024-02-06）: https://arxiv.org/abs/2402.03300
- ORPO（2024-03-12）: https://arxiv.org/abs/2403.07691
- SimPO（2024-05-23）: https://arxiv.org/abs/2405.14734
- chiPO（2024-07-18）: https://arxiv.org/abs/2407.13399
- GDPO（EMNLP 2024）: https://aclanthology.org/2024.emnlp-main.951/
- GiGPO（2025-05-15）: https://arxiv.org/abs/2505.10978
- GSPO（2025-07-24）: https://arxiv.org/abs/2507.18071
- SAPO（2025-11-25）: https://arxiv.org/abs/2511.20347